In [1]:
# environment setup
# %pip install -q transformers datasets accelerate sentencepiece protobuf

In [2]:
import csv
import os
import random
import re
from pathlib import Path
import torch

from datasets import Dataset
from transformers import (
    AutoModelForSeq2SeqLM,
    AutoTokenizer,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    set_seed,
)

os.environ["WANDB_DISABLED"] = "true"

MODEL_NAME = "google/flan-t5-small"
OUTPUT_DIR = "outputs/flan-t5-small-idiom-flute"

SEED = 42
MAX_INPUT_LENGTH = 256
MAX_TARGET_LENGTH = 128
TRAIN_EPOCHS = 50
LEARNING_RATE = 1e-4
BATCH_SIZE = 2

# Keep this small for quick smoke tests. Set to None to train on every CSV row.
MAX_REAL_ROWS = None
INCLUDE_GENERATION_TASK = True
TEST_FRACTION = 0.2
SAMPLE_EXAMPLES_PER_TASK = 2
SANITY_CHECK_EXAMPLES = 2

set_seed(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

Using device: cuda


In [3]:
# data loading

from google.colab import drive
drive.mount('/content/drive')

CSV_CANDIDATES = [
    Path('/content/drive/MyDrive/CSED504/flute_idioms.csv'),
    Path("data/flute_idioms.csv"),
    Path("flute_idioms.csv"),
]

data_csv = next((path for path in CSV_CANDIDATES if path.exists()), None)


def mask_idiom(text, idiom, mask="[IDIOM]"):
    escaped_idiom = re.escape(idiom.strip())
    return re.sub(rf"(?<!\w){escaped_idiom}(?!\w)", mask, text, flags=re.IGNORECASE)


def make_interpretation_example(row):
    idiom = row["idiom"].strip()
    sentence = row["example"].strip()
    meaning = row["explanation_correct"].strip()
    prompt = (
        f"Idiom: {idiom}\n"
        f"Sentence: {sentence}"
    )
    return {
        "task": "interpretation",
        "input": prompt,
        "target": meaning,
    }


def make_generation_example(row):
    idiom = row["idiom"].strip()
    sentence = row["example"].strip()
    meaning = row["explanation_correct"].strip()
    scenario = row.get("correct_substitution", "").strip() or sentence
    masked_meaning = mask_idiom(meaning, idiom)
    masked_scenario = mask_idiom(scenario, idiom)
    prompt = (
        f"Meaning: {masked_meaning}\n"
        f"Scenario: {masked_scenario}"
    )
    return {
        "task": "generation",
        "input": prompt,
        "target": f"Idiom: {idiom}\nSentence: {sentence}",
    }


if data_csv is None:
    expected_paths = ", ".join(str(path) for path in CSV_CANDIDATES)
    raise FileNotFoundError(f"Could not find formatted FLUTE CSV. Expected one of: {expected_paths}")

with data_csv.open("r", encoding="utf-8-sig", newline="") as csv_file:
    rows = list(csv.DictReader(csv_file))

required_columns = {"idiom", "example", "explanation_correct"}
missing_columns = required_columns - set(rows[0].keys() if rows else [])
if missing_columns:
    raise ValueError(f"CSV is missing required columns: {sorted(missing_columns)}")

rows = [row for row in rows if all(row.get(column, "").strip() for column in required_columns)]
if MAX_REAL_ROWS is not None:
    rows = rows[:MAX_REAL_ROWS]

if len(rows) < 2:
    raise ValueError("Need at least 2 usable CSV rows to make a train/test split.")

split_rows = rows.copy()
random.Random(SEED).shuffle(split_rows)

test_row_count = max(1, round(len(split_rows) * TEST_FRACTION))
test_row_count = min(test_row_count, len(split_rows) - 1)
test_rows = split_rows[:test_row_count]
train_rows = split_rows[test_row_count:]


def build_task_examples(source_rows):
    examples = []
    for row in source_rows:
        examples.append(make_interpretation_example(row))
        if INCLUDE_GENERATION_TASK:
            examples.append(make_generation_example(row))
    return examples


train_examples = build_task_examples(train_rows)
test_examples = build_task_examples(test_rows)

print(f"Loaded {len(rows)} usable CSV rows from {data_csv}")
print(f"Split into {len(train_rows)} training rows and {len(test_rows)} held-out test rows")
print(f"Built {len(train_examples)} training examples and {len(test_examples)} held-out test examples")

train_dataset = Dataset.from_list(train_examples)
test_dataset = Dataset.from_list(test_examples)
test_interpretation_examples = [example for example in test_examples if example["task"] == "interpretation"]
test_generation_examples = [example for example in test_examples if example["task"] == "generation"]

preview_interpretation_examples = test_interpretation_examples[:SAMPLE_EXAMPLES_PER_TASK]
preview_generation_examples = test_generation_examples[:SAMPLE_EXAMPLES_PER_TASK]
train_dataset

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Loaded 884 usable CSV rows from /content/drive/MyDrive/CSED504/flute_idioms.csv
Split into 707 training rows and 177 held-out test rows
Built 1414 training examples and 354 held-out test examples


Dataset({
    features: ['task', 'input', 'target'],
    num_rows: 1414
})

In [4]:
# load FLAN-T5-Small
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME).to(device)

print(f"Loaded {MODEL_NAME}")

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


Loaded google/flan-t5-small


In [5]:
# sampling

def generate_text(prompt, max_new_tokens=96, do_sample=False, temperature=0.7):
    encoded = tokenizer(
        prompt,
        return_tensors="pt",
        max_length=MAX_INPUT_LENGTH,
        truncation=True,
    ).to(model.device)

    generation_kwargs = {
        "max_new_tokens": max_new_tokens,
        "num_beams": 4,
        "do_sample": do_sample,
    }
    if do_sample:
        generation_kwargs["temperature"] = temperature
        generation_kwargs["top_p"] = 0.9

    model.eval()
    with torch.no_grad():
        output_ids = model.generate(**encoded, **generation_kwargs)

    return tokenizer.decode(output_ids[0], skip_special_tokens=True)


def sample_examples(title, examples):
    print("=" * 80)
    print(title)
    print("=" * 80)
    if not examples:
        print("No examples available for this task.")
        return
    for idx, example in enumerate(examples, start=1):
        print(f"\nExample {idx}: {example['task']}")
        print("-" * 80)
        print("PROMPT:")
        print(example["input"])
        print("\nMODEL OUTPUT:")
        print(generate_text(example["input"]))
        print("\nTARGET:")
        print(example["target"])


sample_examples("Before fine-tuning: interpretation", preview_interpretation_examples)

Before fine-tuning: interpretation

Example 1: interpretation
--------------------------------------------------------------------------------
PROMPT:
Idiom: chickens come home to roost
Sentence: Short answer: because she was bitten and chickens come home to roost eventually.

MODEL OUTPUT:
because she was bitten

TARGET:
The idiom chickens come home to roost means that you have to face the consequences of your mistakes or bad deeds, and this is what the entailment is saying.

Example 2: interpretation
--------------------------------------------------------------------------------
PROMPT:
Idiom: get cracking
Sentence: Lots of hungry mouths to feed again, so lets get cracking.

MODEL OUTPUT:
Idiom will get cracking.

TARGET:
To get cracking means to start working on something, which is what they need to do in order to feed all the hungry mouths.


In [6]:
# sample generation examples before fine-tuning
sample_examples("Before fine-tuning: generation", preview_generation_examples)

Before fine-tuning: generation

Example 1: generation
--------------------------------------------------------------------------------
PROMPT:
Meaning: The idiom [IDIOM] means that you have to face the consequences of your mistakes or bad deeds, and this is what the entailment is saying.
Scenario: Short answer: because she was bitten and she will have to face the consequences of her mistakes eventually.

MODEL OUTPUT:
because she was bitten and she will have to face the consequences of her mistakes eventually.

TARGET:
Idiom: chickens come home to roost
Sentence: Short answer: because she was bitten and chickens come home to roost eventually.

Example 2: generation
--------------------------------------------------------------------------------
PROMPT:
Meaning: To [IDIOM] means to start working on something, which is what they need to do in order to feed all the hungry mouths.
Scenario: Let's start working on this so that we can feed all the hungry mouths again.

MODEL OUTPUT:
Summary:

In [7]:
#tokenize data

def preprocess_batch(batch):
    model_inputs = tokenizer(
        batch["input"],
        max_length=MAX_INPUT_LENGTH,
        truncation=True,
    )

    labels = tokenizer(
        text_target=batch["target"],
        max_length=MAX_TARGET_LENGTH,
        truncation=True,
    )

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs


tokenized_train = train_dataset.map(
    preprocess_batch,
    batched=True,
    remove_columns=train_dataset.column_names,
)

data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
)

tokenized_train

Map:   0%|          | 0/1414 [00:00<?, ? examples/s]

Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 1414
})

In [8]:
# sanity-check labels and loss before training
# Use a tiny debug batch here. Running this over the full training set can cause CUDA OOM.
debug_count = min(SANITY_CHECK_EXAMPLES, len(tokenized_train))
debug_indices = list(range(debug_count))
debug_batch = data_collator([tokenized_train[i] for i in debug_indices])
debug_batch = {key: value.to(model.device) for key, value in debug_batch.items()}

non_padding_target_tokens = (debug_batch["labels"] != -100).sum().item()
print(f"Non-padding target tokens: {non_padding_target_tokens}")

for idx in debug_indices:
    print(f"\nDecoded target {idx + 1}:")
    print(tokenizer.decode(tokenized_train[idx]["labels"], skip_special_tokens=True))

model.eval()
with torch.no_grad():
    initial_loss = model(**debug_batch).loss.item()

print(f"\nInitial loss before fine-tuning: {initial_loss:.4f}")
assert non_padding_target_tokens > 0, "No target tokens found. Check preprocessing."
assert torch.isfinite(torch.tensor(initial_loss)), "Initial loss is not finite."
assert initial_loss > 0, "Initial loss should usually be nonzero before training."


Non-padding target tokens: 67

Decoded target 1:
Nuts and bolts refer to the most basic and essential aspects of something, which is what was filling the drawers.

Decoded target 2:
Idiom: nuts and bolts Sentence: The four foot tall shelf of endless drawers filled with nuts and bolts that beau hadn't finished sorting.

Initial loss before fine-tuning: 2.3452


In [10]:
# fine-tuning
training_args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR,
    learning_rate=LEARNING_RATE,
    per_device_train_batch_size=BATCH_SIZE,
    num_train_epochs=TRAIN_EPOCHS,
    weight_decay=0.0,
    logging_steps=1000,
    save_strategy="no",
    report_to="none",
    # Keep fp16 off for this tiny demo. In this environment it caused 0.0 loss and nan gradients.
    fp16=False,
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    data_collator=data_collator,
    processing_class=tokenizer,
)

trainer.train()

Step,Training Loss
1000,0.901059
2000,0.908081
3000,0.849467
4000,0.733630
5000,0.657125
6000,0.568746
7000,0.517420
8000,0.455687
9000,0.410498
10000,0.371687


TrainOutput(global_step=35350, training_loss=0.2877716130819145, metrics={'train_runtime': 3551.7355, 'train_samples_per_second': 19.906, 'train_steps_per_second': 9.953, 'total_flos': 1470126674608128.0, 'train_loss': 0.2877716130819145, 'epoch': 50.0})

In [17]:
# sample interpretation examples after fine-tuning
sample_examples("After fine-tuning: interpretation", preview_interpretation_examples)

After fine-tuning: interpretation

Example 1: interpretation
--------------------------------------------------------------------------------
PROMPT:
Idiom: chickens come home to roost
Sentence: Short answer: because she was bitten and chickens come home to roost eventually.

MODEL OUTPUT:
To be bitten means to be bitten by someone, so the entailment is that she was bitten by someone.

TARGET:
The idiom chickens come home to roost means that you have to face the consequences of your mistakes or bad deeds, and this is what the entailment is saying.

Example 2: interpretation
--------------------------------------------------------------------------------
PROMPT:
Idiom: get cracking
Sentence: Lots of hungry mouths to feed again, so lets get cracking.

MODEL OUTPUT:
To get cracking means to start working on something, so the imperative form would be to start working on something.

TARGET:
To get cracking means to start working on something, which is what they need to do in order to feed a

In [15]:
# sample generation examples after fine-tuning
sample_examples("After fine-tuning: generation", preview_generation_examples)

After fine-tuning: generation

Example 1: generation
--------------------------------------------------------------------------------
PROMPT:
Meaning: The idiom [IDIOM] means that you have to face the consequences of your mistakes or bad deeds, and this is what the entailment is saying.
Scenario: Short answer: because she was bitten and she will have to face the consequences of her mistakes eventually.

MODEL OUTPUT:
Idiom: chickens come home Sentence: Short answer: because she was bitten and she will have to chickens come home.

TARGET:
Idiom: chickens come home to roost
Sentence: Short answer: because she was bitten and chickens come home to roost eventually.

Example 2: generation
--------------------------------------------------------------------------------
PROMPT:
Meaning: To [IDIOM] means to start working on something, which is what they need to do in order to feed all the hungry mouths.
Scenario: Let's start working on this so that we can feed all the hungry mouths again.

MOD

In [16]:
# save
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

print(f"Saved fine-tuned model and tokenizer to: {OUTPUT_DIR}")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved fine-tuned model and tokenizer to: outputs/flan-t5-small-idiom-flute


In [33]:
print(preview_generation_examples)
print(preview_interpretation_examples)

[{'task': 'generation', 'input': 'Meaning: The idiom [IDIOM] means that you have to face the consequences of your mistakes or bad deeds, and this is what the entailment is saying.\nScenario: Short answer: because she was bitten and she will have to face the consequences of her mistakes eventually.', 'target': 'Idiom: chickens come home to roost\nSentence: Short answer: because she was bitten and chickens come home to roost eventually.'}, {'task': 'generation', 'input': "Meaning: To [IDIOM] means to start working on something, which is what they need to do in order to feed all the hungry mouths.\nScenario: Let's start working on this so that we can feed all the hungry mouths again.", 'target': 'Idiom: get cracking\nSentence: Lots of hungry mouths to feed again, so lets get cracking.'}]
[{'task': 'interpretation', 'input': 'Idiom: chickens come home to roost\nSentence: Short answer: because she was bitten and chickens come home to roost eventually.', 'target': 'The idiom chickens come ho

In [73]:
custom_generation_example = [{'task': 'generation',
                              'input': (
                                  'Meaning: The idiom [IDIOM] means to do very well, such as turning in a great class project.\n'
                                  'Scenario: Alex, Jonathan, and Marcus performed exceptionally well on their NLP project last week and everyone was very impressed.'
                                  ),
                              'target': ''}]
custom_interpretation_example = [{'task': 'interpretation',
                              'input': (
                                  'Idiom: Lit the scoreboard.\n'
                                  'Sentence: Jonathan, Alex, and Marcus lit the scoreboard in their NLP project last week and the everyone was very impressed.'
                                  ),
                              'target': ''}]

In [74]:
sample_examples('Custom Idiom', custom_generation_example)

Custom Idiom

Example 1: generation
--------------------------------------------------------------------------------
PROMPT:
Meaning: The idiom [IDIOM] means to do very well, such as turning in a great class project.
Scenario: Alex, Jonathan, and Marcus performed exceptionally well on their NLP project last week and everyone was very impressed.

MODEL OUTPUT:
Idiom: tickle the ivories Sentence: Alex, jonathan, and bruce performed tickle the ivories on their NLP project last week and everyone was tickle the ivories.

TARGET:



In [75]:
sample_examples('Custom Idiom', custom_interpretation_example)

Custom Idiom

Example 1: interpretation
--------------------------------------------------------------------------------
PROMPT:
Idiom: Lit the scoreboard.
Sentence: Jonathan, Alex, and Marcus lit the scoreboard in their NLP project last week and the everyone was very impressed.

MODEL OUTPUT:
To Lit the scoreboard means to score a very high score, which is what happened with Jonathan, Alex, and Marcus in their project last week.

TARGET:

